# EDA

**Autor:** duac1011 · **Datum:** 2026-09-28

**Ziel:** Durch eine domänenspezifische explorative Datenanalyse und begleitende Recherche werden die Trainingsdaten im Kontext der Fernerkundung und landwirtschaftlicher Kulturen besser verstanden. <br>
Untersucht werden Struktur, Merkmale und Zielvariablen sowie deren fachliche Zusammenhänge.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_classif
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from awp2.config import (
    AEZ_COL,
    CROP_COL,
    CV_FOLDS,
    FIGURES_DIR,
    ID_COL,
    MONTH_COL,
    STAGE_COL,
    VAL_SIZE,
)
from awp2.data import band_columns, load_test, load_train, wavelengths
from awp2.plots import plot_spectra, save_doc_figure

print(f"Analyseabbildungen: {FIGURES_DIR}")

## 2. Trainingsdaten laden

Die validierten Trainingsdaten laden und die Spektralbandspalten in Wellenlängenreihenfolge bestimmen.

In [ ]:
train = load_train()
test = load_test()
bands = band_columns(train)
available_bands = train[bands].columns[train[bands].notna().any()].tolist()
input_columns = [AEZ_COL, MONTH_COL, *bands]

pd.Series(
    {
        "Trainingszeilen": len(train),
        "Testzeilen": len(test),
        "Eingabespalten": len(input_columns),
        "Spektralbänder": len(bands),
        "vollständig belegte oder teilweise belegte Bänder": len(available_bands),
    }
).to_frame("Anzahl")

## 3. Datenqualität und wiederholte Beobachtungen

**Frage:** Sind die Eingaben in Train und Test technisch vergleichbar, und enthalten einzelne Bänder oder Zeilen keine Messinformation?

Dafür zählen wir fehlende Werte je Band und pro Spektrum, prüfen Wertebereiche sowie konstante Bänder und suchen identische Eingaben. Vollständig leere Bänder können keine Information tragen. Identische oder sehr ähnliche Eingaben können einen zufälligen Train-/Validierungs-Split zu optimistisch machen. Die Ursache einer Lücke lässt sich aus der Tabelle allein jedoch nicht bestimmen.

In [ ]:
train_missing = train[bands].isna().sum()
test_missing = test[bands].isna().sum()
wavelength_nm = np.asarray(wavelengths(train))
fully_empty_bands = train_missing[train_missing.eq(len(train))].index.tolist()
partially_missing = train_missing[train_missing.between(1, len(train) - 1)].sort_values(
    ascending=False
)
constant_bands = train[available_bands].nunique(dropna=True)
constant_bands = constant_bands[constant_bands.le(1)].index.tolist()
all_reflectance = train[available_bands].to_numpy(dtype=float)

quality_summary = pd.DataFrame(
    {
        "Train": [
            len(fully_empty_bands),
            len(partially_missing),
            int(train[available_bands].isna().any(axis=1).sum()),
            len(constant_bands),
            float(np.nanmin(all_reflectance)),
            float(np.nanmax(all_reflectance)),
        ],
        "Test": [
            int(test_missing.eq(len(test)).sum()),
            int(test_missing.between(1, len(test) - 1).sum()),
            int(test[available_bands].isna().any(axis=1).sum()),
            int(test[available_bands].nunique(dropna=True).le(1).sum()),
            float(np.nanmin(test[available_bands].to_numpy(dtype=float))),
            float(np.nanmax(test[available_bands].to_numpy(dtype=float))),
        ],
    },
    index=[
        "vollständig leere Bänder",
        "teilweise fehlende Bänder",
        "Zeilen mit Lücken in verfügbaren Bändern",
        "konstante verfügbare Bänder",
        "minimale Reflektanz [%]",
        "maximale Reflektanz [%]",
    ],
)
display(quality_summary)

schema_columns = [AEZ_COL, MONTH_COL, *bands, CROP_COL, STAGE_COL]
schema_summary = pd.DataFrame(
    {
        "Train": train.dtypes.astype(str),
        "Test": test.dtypes.astype(str),
    }
).reindex(schema_columns)
display(schema_summary)
display(
    pd.DataFrame(
        {
            "Train": [train.index.name == ID_COL, train.index.is_unique],
            "Test": [test.index.name == ID_COL, test.index.is_unique],
        },
        index=["Index heißt id", "Index ist eindeutig"],
    )
)

reflectance_summary = pd.DataFrame(
    {
        "Train": train[available_bands].stack().quantile([0, 0.5, 1]),
        "Test": test[available_bands].stack().quantile([0, 0.5, 1]),
    }
).rename(index={0.0: "Minimum", 0.5: "Median", 1.0: "Maximum"})
display(reflectance_summary)

missing_bands = pd.DataFrame(
    {"Train: fehlend": train_missing, "Test: fehlend": test_missing}
).query("`Train: fehlend` > 0 or `Test: fehlend` > 0")
display(missing_bands)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.scatter(wavelength_nm, train_missing, s=18, label="Train")
ax.scatter(wavelength_nm, test_missing, s=18, marker="x", label="Test")
ax.set(xlabel="Wellenlänge [nm]", ylabel="fehlende Werte", title="Fehlende Werte je Band")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

partially_missing_rows = train.loc[
    train[available_bands].isna().any(axis=1),
    [AEZ_COL, MONTH_COL, CROP_COL, STAGE_COL],
].copy()
partially_missing_rows["fehlende verfügbare Bandwerte"] = (
    train.loc[partially_missing_rows.index, available_bands].isna().sum(axis=1)
)
display(partially_missing_rows.sort_values("fehlende verfügbare Bandwerte", ascending=False))
display(
    partially_missing_rows.groupby([AEZ_COL, MONTH_COL, CROP_COL, STAGE_COL], observed=True)
    .agg(
        Zeilen=("fehlende verfügbare Bandwerte", "size"),
        Fehlende_Bandwerte=("fehlende verfügbare Bandwerte", "sum"),
    )
    .sort_values(["Fehlende_Bandwerte", "Zeilen"], ascending=False)
)

duplicate_mask = train.duplicated(input_columns, keep=False)
duplicate_rows = train.loc[duplicate_mask, [AEZ_COL, MONTH_COL, CROP_COL, STAGE_COL]].copy()
duplicate_rows["fehlende Bandwerte"] = train.loc[duplicate_mask, bands].isna().sum(axis=1)
display(duplicate_rows)

train_inputs = train.reset_index(names="train_id")[input_columns + ["train_id"]]
test_inputs = test.reset_index(names="test_id")[input_columns + ["test_id"]]
train_test_matches = train_inputs.merge(test_inputs, on=input_columns, how="inner")
print(
    "Identische Eingaben inklusive AEZ und Month zwischen Train und Test: "
    f"{len(train_test_matches)}"
)

train_spectra = train.reset_index(names="train_id")[bands + ["train_id", AEZ_COL, MONTH_COL]]
test_spectra = test.reset_index(names="test_id")[bands + ["test_id", AEZ_COL, MONTH_COL]]
spectral_matches = train_spectra.merge(
    test_spectra, on=bands, how="inner", suffixes=("_train", "_test")
)
display(spectral_matches)

In [ ]:
missing_group_columns = [AEZ_COL, MONTH_COL, CROP_COL, STAGE_COL]
group_sizes = train.groupby(missing_group_columns, observed=True).size().rename("Zeilen insgesamt")
missing_group_summary = (
    partially_missing_rows.groupby(missing_group_columns, observed=True)
    .agg(
        Zeilen_mit_Lücken=("fehlende verfügbare Bandwerte", "size"),
        Fehlende_Bandwerte=("fehlende verfügbare Bandwerte", "sum"),
    )
    .join(group_sizes)
    .assign(
        Anteil_Zeilen_mit_Lücken=lambda frame: (
            frame["Zeilen_mit_Lücken"] / frame["Zeilen insgesamt"]
        )
    )
    .sort_values(["Anteil_Zeilen_mit_Lücken", "Zeilen_mit_Lücken"], ascending=False)
)
display(missing_group_summary)

spectrum_metrics = pd.concat(
    {
        "Train": pd.DataFrame(
            {
                "Median je Spektrum": train[available_bands].median(axis=1),
                "Interquartilsabstand je Spektrum": train[available_bands].quantile(0.75, axis=1)
                - train[available_bands].quantile(0.25, axis=1),
            }
        ),
        "Test": pd.DataFrame(
            {
                "Median je Spektrum": test[available_bands].median(axis=1),
                "Interquartilsabstand je Spektrum": test[available_bands].quantile(0.75, axis=1)
                - test[available_bands].quantile(0.25, axis=1),
            }
        ),
    },
    names=["Datensatz"],
)
display(spectrum_metrics.groupby(level="Datensatz").describe())

In [ ]:
save_doc_figure(fig, "fehlende_werte_baender")

### Beobachtung zur Datenqualität

- In Train und Test sind jeweils 67 Bänder vollständig leer; damit bleiben 131 Bänder mit mindestens einer Messung.
- Zusätzlich fehlen in Train Werte in sechs und in Test in drei verfügbaren Bändern. Das betrifft 44 von 5.591 Trainings- und 11 von 1.397 Testzeilen.
- Beide Datensätze verwenden `id` als eindeutigen Index. `AEZ` und `Month` sind jeweils `int64`, die Bandwerte jeweils `float64`; `Crop` und `Stage` kommen erwartungsgemäß nur in Train vor.
- Die verfügbaren Bänder sind nicht konstant. Ihr globaler Reflektanzmedian liegt bei 24,68 % in Train und 24,64 % in Test; auch die zeilenweisen Mediane und Interquartilsabstände sind ähnlich verteilt.
- Die 44 Trainingszeilen mit partiellen Lücken verteilen sich auf mehrere AEZ-Monat-Crop-Stage-Gruppen. Die höchste beobachtete Gruppenrate beträgt 2 von 42 Zeilen (4,76 %) für `AEZ=9`, Monat 9, Baumwolle im Stadium `Critical`; die meisten absoluten Lücken liegen bei `AEZ=10`, Monat 8, Soja im Stadium `Critical` (10 von 231 Zeilen).
- Zwei Gruppen mit jeweils zwei Trainingszeilen haben exakt gleiche Eingaben einschließlich `AEZ` und `Month`. Zwischen Train und Test gibt es weder identische vollständige Eingaben noch identische Spektren.

Damit sind vollständig leere Bänder und Trainingsduplikate konkrete Datenqualitätsbefunde. Die partiellen Lücken und die robusten Spektralkennwerte liefern keinen Beweis für eine technische Ursache oder gegen jede mögliche Verteilungsverschiebung. Sie zeigen nur, dass die beobachteten Lücken in einzelnen Gruppen etwas häufiger auftreten und die ausgewählten Train-Test-Kennwerte ähnlich sind. Der Befund zu Train-Test-Treffern weicht von einem bisherigen Hinweis in der Pipeline-Dokumentation ab und sollte vor einer Cleaning-Entscheidung abgeglichen werden.

## 4. Labels und beobachtete Crop-Stage-Paare

**Frage:** Welche Klassen sind selten, und welche Kombinationen von Kultur und Stadium kommen in der Trainingsstichprobe überhaupt vor?

Wir zählen `Crop`, `Stage` und ihre Kombinationen. Das beantwortet die Stichprobenfrage und zeigt, ob ein zufälliger Split seltene Gruppen verlieren könnte. Es beantwortet nicht, ob eine im Training fehlende Kombination landwirtschaftlich unmöglich ist; dafür fehlen Angaben zur Auswahl und Labelbildung.

In [ ]:
crop_counts = train[CROP_COL].value_counts()
stage_counts = train[STAGE_COL].value_counts()
pair_counts = train.groupby([CROP_COL, STAGE_COL], observed=True).size().sort_values()
pair_table = pd.crosstab(train[CROP_COL], train[STAGE_COL])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
crop_counts.sort_values().plot.barh(ax=axes[0], color="C0", title="Crop")
stage_counts.sort_values().plot.barh(ax=axes[1], color="C1", title="Stage")
for ax in axes:
    ax.set_xlabel("Anzahl Trainingszeilen")
    ax.grid(axis="x", alpha=0.3)
fig.suptitle("Klassenverteilungen")
fig.tight_layout()
plt.show()

display(pair_table)
display(
    pair_counts.rename("Anzahl")
    .to_frame()
    .assign(Anteil=lambda frame: frame["Anzahl"] / len(train))
)

conservative_train_counts = np.floor(pair_counts * (1 - VAL_SIZE)).astype(int)
conservative_validation_counts = np.floor(pair_counts * VAL_SIZE).astype(int)
pair_split_check = pd.DataFrame(
    {
        "Beobachtungen insgesamt": pair_counts,
        "mindestens im 70%-Training": conservative_train_counts,
        "mindestens in 30%-Validierung": conservative_validation_counts,
        "Fünf CV-Folds im Training möglich": conservative_train_counts.ge(CV_FOLDS),
    }
)
display(pair_split_check)

In [ ]:
save_doc_figure(fig, "klassenverteilungen")

### Beobachtung zu Labels

- `corn` (2.097 Zeilen) und `soybean` (1.669) dominieren die Crop-Labels; `rice` ist mit 93 Zeilen die seltenste Kultur.
- `Harvest` ist mit 180 Zeilen das seltenste Stadium. Das kleinste beobachtete Crop-Stage-Paar ist `cotton|Harvest` mit 11 Zeilen.
- Mehrere Paare haben in der Trainingsstichprobe die Häufigkeit null, beispielsweise alle Reis-Paare außer `Early_Mid` und `Late`.
- Die konservative Zählprüfung für den vorgesehenen 70/30-Split lässt für `cotton|Harvest` mindestens sieben Zeilen im Training und drei in der Validierung erwarten. Alle beobachteten Paare haben danach mindestens fünf Trainingszeilen und können daher in fünf inneren CV-Folds vertreten sein.

Ein späterer Split muss deshalb nach dem kombinierten Label `Crop|Stage` stratifizieren, und eine klassenfaire Kennzahl wie Balanced Accuracy ist notwendig. Die Zählprüfung garantiert keine stabile Leistungsbewertung für seltene Paare: Schon wenige falsch klassifizierte Beispiele können ihre Kennzahlen stark verändern. Die Kreuztabelle ist keine Fachregel; sie beschreibt nur, was in dieser Auswahl beobachtet wurde.

## 5. Metadaten: AEZ und Aufnahmemonat

**Frage:** Sind `AEZ` und `Month` zwischen Train und Test ähnlich verteilt, und strukturieren sie `Crop` oder `Stage` bereits stark?

`AEZ` fasst Gebiete mit ähnlichen Bedingungen zusammen; `Month` kann phänologische Information enthalten. Beides könnte hilfreicher Kontext oder ein Shortcut sein, bei dem ein späteres Modell vor allem Ort oder Monat statt des Spektrums nutzt. Die folgenden Tabellen zeigen die Verteilungen, beweisen aber weder Kausalität noch die genaue Bedeutung der Codes.

In [ ]:
aez_crop_counts = pd.crosstab(train[AEZ_COL], train[CROP_COL])
aez_crop_share = aez_crop_counts.div(aez_crop_counts.sum(axis=1), axis=0)
metadata_distribution = pd.concat(
    {
        "Train": train[[AEZ_COL, MONTH_COL]],
        "Test": test[[AEZ_COL, MONTH_COL]],
    },
    names=["Datensatz"],
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
aez_crop_share.plot.bar(stacked=True, ax=axes[0], colormap="tab10")
axes[0].set(
    xlabel="AEZ",
    ylabel="Anteil innerhalb der AEZ",
    title="Crop-Anteile je AEZ",
)
axes[0].legend(title=CROP_COL, fontsize="small")
axes[0].grid(axis="y", alpha=0.3)

for dataset_name, frame in metadata_distribution.groupby(level="Datensatz"):
    axes[1].hist(
        frame[MONTH_COL],
        bins=np.arange(frame[MONTH_COL].min() - 0.5, frame[MONTH_COL].max() + 1.5),
        alpha=0.55,
        label=dataset_name,
    )
axes[1].set(xlabel="Month", ylabel="Anzahl", title="Monatsverteilung: Train und Test")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

dataset_names = metadata_distribution.index.get_level_values("Datensatz")
display(pd.crosstab(dataset_names, metadata_distribution[AEZ_COL]))
display(pd.crosstab(dataset_names, metadata_distribution[MONTH_COL]))

month_stage = pd.crosstab(
    [train[CROP_COL], train[MONTH_COL]], train[STAGE_COL], normalize="index"
).sort_index()
display(month_stage)

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharex=True, sharey=True)
for ax, (crop, crop_data) in zip(axes.flat, train.groupby(CROP_COL, observed=True), strict=False):
    pd.crosstab(crop_data[MONTH_COL], crop_data[STAGE_COL]).plot.bar(
        stacked=True, ax=ax, colormap="tab20"
    )
    ax.set(title=crop, xlabel="Month", ylabel="Anzahl")
    ax.legend(fontsize="x-small", title=STAGE_COL)
    ax.grid(axis="y", alpha=0.3)
axes[-1, -1].set_visible(False)
fig.suptitle("Stage-Häufigkeiten je Monat innerhalb jeder Kultur", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
save_doc_figure(fig, "stadien_monat_crop")

### Beobachtung zu Metadaten

- In Train und Test kommen dieselben sieben AEZ-Werte sowie die Monate 5 bis 10 vor. Die Häufigkeiten sind wegen der unterschiedlichen Stichprobengröße nicht gleich, die Kategorien überlappen aber vollständig.
- Die Crop-Anteile unterscheiden sich deutlich zwischen AEZ. Einzelne Zonen enthalten in dieser Stichprobe nur eine Kultur, daher kann `AEZ` eine Kultur stark vorstrukturieren.
- Innerhalb einzelner Kulturen konzentrieren sich die Stage-Labels oft auf wenige Monate. Bei Mais besteht Monat 8 im Training ausschließlich aus `Critical`.

`AEZ` und insbesondere `Month` sind damit potenziell nützliche, aber auch fragile Kontextmerkmale. Die Tabellen zeigen eine Stichprobenassoziation, keine fachlich bestätigte Definition der Stadien und keinen Beweis, dass die Metadaten beim späteren Einsatz verfügbar sind.

## 5. Spektrale Signaturen je Klasse (#17)

**Frage:** Unterscheiden sich die mittleren Spektren der Kulturen, und bleiben Unterschiede sichtbar, wenn wir nur innerhalb einer Kultur nach `Stage` vergleichen?

Ein Spektrum beschreibt die Reflektanz über die Wellenlänge, nicht über die Zeit. Wir zeigen Mittelwert und Streuung nach Kultur sowie mittlere Stadien-Spektren innerhalb jeder Kultur. Besonders interessant sind sichtbares Licht (VIS), der Übergang von Rot zu NIR (Red Edge), NIR und SWIR. Ein sichtbarer Abstand ist zunächst ein explorativer Hinweis; er kann auch durch Bodenanteil, Aufnahmebedingungen, AEZ, Monat oder ungleiche Gruppengrößen entstehen.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_spectra(
    train,
    ax=axes[0],
    title="Mittleres Spektrum aller Trainingsdaten (± 1 Std)",
)
plot_spectra(
    train,
    by=CROP_COL,
    ax=axes[1],
    title="Mittlere Spektren nach Kultur (± 1 Std)",
)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
for ax, (crop, crop_data) in zip(axes.flat, train.groupby(CROP_COL, observed=True), strict=False):
    plot_spectra(
        crop_data,
        by=STAGE_COL,
        std=False,
        ax=ax,
        title=f"{crop}: mittlere Spektren je Stage",
    )
axes[-1, -1].set_visible(False)
fig.suptitle("Stadien nur innerhalb derselben Kultur verglichen", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
save_doc_figure(fig, "stadien_spektren_crop")

### Beobachtung zu spektralen Signaturen

- Die mittleren Kurven unterscheiden sich zwischen den Kulturen in mehreren Bereichen der Spektralachse; die leeren Bänder erscheinen als gemeinsame Lücken.
- Auch innerhalb derselben Kultur liegen die mittleren Kurven der beobachteten Stadien in mehreren Bereichen auseinander. Dies ist beispielsweise bei Mais und Soja im NIR und SWIR sichtbar.
- Die Streuungsbereiche der Kulturen überlappen jedoch teilweise deutlich. Eine Klassenbezeichnung lässt sich daher nicht aus einer einzelnen Kurve oder einem einzelnen Band ableiten.

Die Plots stützen die Hypothese, dass die vollständige Spektralform Information für `Crop` und `Stage` enthält. Sie belegen noch keine Vorhersagequalität und trennen nicht zwischen phänologischen Effekten, Metadaten, Mischpixeln und Aufnahmebedingungen.

### Vegetationsindizes

**Frage:** Fassen einfache, fachlich begründete Verhältnisse sichtbare Unterschiede zwischen Kulturen oder Stadien zusammen?

NDVI, NDRE, PRI und NDWI kombinieren jeweils zwei nahegelegene, verfügbare Bänder. Sie erzeugen keine neue Messinformation, machen die Spektren aber in einer kleineren Anzahl verständlicher Kennzahlen vergleichbar. Die Verteilungen zeigen nur mögliche Gruppenunterschiede; ein Nutzen für eine Klassifikation muss später separat und ohne Leakage validiert werden.

In [ ]:
RED_BAND = "X671"
NIR_BAND = "X854"
RED_EDGE_BAND = "X702"
PRI_FIRST_BAND = "X529"
PRI_SECOND_BAND = "X569"
SWIR_BAND = "X1235"
INDEX_BANDS = [
    RED_BAND,
    NIR_BAND,
    RED_EDGE_BAND,
    PRI_FIRST_BAND,
    PRI_SECOND_BAND,
    SWIR_BAND,
]

reflectance = train[INDEX_BANDS] / 100
vegetation_indices = pd.DataFrame(
    {
        "NDVI": (reflectance[NIR_BAND] - reflectance[RED_BAND])
        / (reflectance[NIR_BAND] + reflectance[RED_BAND]),
        "NDRE": (reflectance[NIR_BAND] - reflectance[RED_EDGE_BAND])
        / (reflectance[NIR_BAND] + reflectance[RED_EDGE_BAND]),
        "PRI": (reflectance[PRI_FIRST_BAND] - reflectance[PRI_SECOND_BAND])
        / (reflectance[PRI_FIRST_BAND] + reflectance[PRI_SECOND_BAND]),
        "NDWI": (reflectance[NIR_BAND] - reflectance[SWIR_BAND])
        / (reflectance[NIR_BAND] + reflectance[SWIR_BAND]),
    },
    index=train.index,
).join(train[[CROP_COL, STAGE_COL]])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for column, crop_ax, stage_ax in zip(
    ["NDVI", "NDRE", "PRI", "NDWI"], axes[0], axes[1], strict=True
):
    crop_groups = [
        frame[column].dropna().to_numpy()
        for _, frame in vegetation_indices.groupby(CROP_COL, observed=True)
    ]
    crop_labels = list(vegetation_indices.groupby(CROP_COL, observed=True).groups)
    crop_ax.boxplot(crop_groups, tick_labels=crop_labels, showfliers=False)
    crop_ax.set(title=f"{column} nach Crop", ylabel=column)
    crop_ax.tick_params(axis="x", rotation=35)

    stage_groups = [
        frame[column].dropna().to_numpy()
        for _, frame in vegetation_indices.groupby(STAGE_COL, observed=True)
    ]
    stage_labels = list(vegetation_indices.groupby(STAGE_COL, observed=True).groups)
    stage_ax.boxplot(stage_groups, tick_labels=stage_labels, showfliers=False)
    stage_ax.set(title=f"{column} nach Stage", ylabel=column)
    stage_ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
plt.show()

index_columns = ["NDVI", "NDRE", "PRI", "NDWI"]
display(vegetation_indices.groupby(CROP_COL, observed=True)[index_columns].median())
display(vegetation_indices.groupby(STAGE_COL, observed=True)[index_columns].median())

### Beobachtung zu Vegetationsindizes

- Die Mediane von NDVI, NDRE und NDWI unterscheiden sich zwischen den Kulturen. Reis weist in dieser Stichprobe die höchsten Mediane auf, Baumwolle und Winterweizen deutlich niedrigere.
- Nach `Stage` hat `Late` die höchsten Mediane für NDVI, NDRE und NDWI; `Emerge_VEarly` und `Harvest` liegen deutlich niedriger. Die Boxplots zeigen aber auch Überlappungen.
- PRI unterscheidet die Gruppen in diesen Daten weniger klar als die drei anderen Indizes.

Das Muster ist mit Unterschieden in grüner Vegetation, Blattstruktur und Wasserzustand vereinbar. Es ist keine eindeutige physikalische Zuordnung: Kultur, Stadium, AEZ und Monat sind in der Stichprobe nicht unabhängig. Die Indizes bleiben deshalb Kandidaten für einen späteren validierten Vergleich, nicht bereits ausgewählte Modellmerkmale.

### Spektren nach Agro-Ecological Zone (AEZ)

**Frage:** Unterscheiden sich die gemessenen Spektren zwischen den agro-ökologischen Zonen?

AEZ gruppiert Regionen mit ähnlichen Klima-, Boden- und Anbaubedingungen. Unterschiedliche mittlere Spektren wären deshalb erwartbar, könnten aber auch auf abweichende Kultur-, Stadium- oder Monatshäufigkeiten zurückgehen. Der Vergleich ist eine Prüfung auf möglichen Kontext bzw. Shortcut-Learning, kein Nachweis eines kausalen AEZ-Effekts.

In [ ]:
aez_counts = train[AEZ_COL].value_counts().sort_index()
display(aez_counts.rename("Anzahl").to_frame())

fig, ax = plt.subplots(figsize=(12, 5))
plot_spectra(
    train,
    by=AEZ_COL,
    std=False,
    ax=ax,
    title="Mittlere Spektren je AEZ",
)
plt.show()

### Beobachtung zu AEZ-Spektren

- Die sieben AEZ haben unterschiedliche mittlere Spektren; die Unterschiede sind besonders im NIR und SWIR sichtbar.
- Die AEZ-Gruppen sind sehr unterschiedlich groß: von 50 Beobachtungen in AEZ 2 bis 1.872 in AEZ 10.
- Die vorherige Kreuztabelle zeigt zugleich, dass sich die Crop-Anteile je AEZ unterscheiden.

Die Spektren belegen damit eine Assoziation von AEZ und Signal, aber keinen eigenständigen Klima- oder Bodeneffekt. Für die Modellierung bleibt deshalb der Vergleich mit und ohne AEZ notwendig.

## 6. Korrelation und Dimensionalität (#18)

**Frage:** Wie redundant sind benachbarte Spektralbänder, und wie viele Hauptkomponenten beschreiben die Variation der vollständigen Spektren?

Hohe Korrelationen benachbarter Wellenlängen deuten auf redundante Information. Die Hauptkomponentenanalyse (PCA) fasst korrelierte Bänder zu unkorrelierten Komponenten zusammen. Beides zeigt Struktur der Eingaben, ist aber kein Leistungsnachweis für eine spätere Klassifikation.

## 6. Korrelation und Dimensionalität (#18)

**Frage:** Sind benachbarte Bänder weitgehend redundant, an welchen Wellenlängen unterscheiden sich die Labels am stärksten, und zeigt eine verdichtete Darstellung Gruppen?

Wir verwenden nur die 131 nicht vollständig leeren Bänder und für diese Visualisierungen die 5.547 Zeilen ohne zusätzliche Lücken. Die Korrelation benachbarter Bänder beschreibt Redundanz. Der ANOVA-F-Wert misst einen univariaten Unterschied der Gruppenmittelwerte je Band. Die PCA fasst viele gemeinsam variierende Bänder zu Komponenten zusammen. Alle drei Auswertungen sind EDA: Sie wählen keine endgültigen Modellmerkmale aus und die PCA wird später gegebenenfalls ausschließlich im Trainingsanteil gelernt.

In [ ]:
complete_train = train.dropna(subset=available_bands).copy()
complete_spectra = complete_train[available_bands]
available_wavelengths = np.asarray(wavelengths(complete_spectra))

neighbor_correlation = pd.Series(
    np.diag(complete_spectra.corr().to_numpy(), k=1),
    index=[
        f"{left} - {right}"
        for left, right in zip(available_bands[:-1], available_bands[1:], strict=True)
    ],
    name="Korrelation",
)
display(neighbor_correlation.describe().to_frame())

crop_f, _ = f_classif(complete_spectra, complete_train[CROP_COL])
stage_f, _ = f_classif(complete_spectra, complete_train[STAGE_COL])
anova_scores = pd.DataFrame(
    {
        "Wellenlänge [nm]": available_wavelengths,
        "Crop": crop_f,
        "Stage": stage_f,
    }
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(neighbor_correlation.to_numpy(), linewidth=1)
axes[0].set(
    xlabel="Paar benachbarter verfügbarer Bänder",
    ylabel="Pearson-Korrelation",
    title="Korrelation benachbarter Bänder",
)
axes[0].grid(alpha=0.3)
for label in [CROP_COL, STAGE_COL]:
    axes[1].plot(
        anova_scores["Wellenlänge [nm]"],
        anova_scores[label],
        label=label,
        linewidth=1.2,
    )
axes[1].set(
    xlabel="Wellenlänge [nm]",
    ylabel="ANOVA-F-Wert",
    title="Univariate Klassenunterschiede je Band",
)
axes[1].legend()
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

scaled_spectra = StandardScaler().fit_transform(complete_spectra)
pca = PCA(n_components=10)
pca_scores = pca.fit_transform(scaled_spectra)
pca_frame = pd.DataFrame(
    pca_scores[:, :2],
    columns=["PC1", "PC2"],
    index=complete_train.index,
).join(complete_train[[CROP_COL, STAGE_COL]])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(
    np.arange(1, len(pca.explained_variance_ratio_) + 1),
    np.cumsum(pca.explained_variance_ratio_),
    marker="o",
)
axes[0].set(
    xlabel="Anzahl Hauptkomponenten",
    ylabel="kumulierte erklärte Varianz",
    title="PCA: erklärte Varianz",
)
axes[0].set_ylim(0, 1.02)
axes[0].grid(alpha=0.3)
for ax, label in zip(axes[1:], [CROP_COL, STAGE_COL], strict=True):
    for group, frame in pca_frame.groupby(label, observed=True):
        ax.scatter(frame["PC1"], frame["PC2"], s=9, alpha=0.55, label=group)
    ax.set(xlabel="PC1", ylabel="PC2", title=f"PCA nach {label}")
    ax.legend(fontsize="x-small", title=label)
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

display(anova_scores.nlargest(10, CROP_COL))
display(anova_scores.nlargest(10, STAGE_COL))

In [ ]:
save_doc_figure(fig, "pca_projektion")

### Beobachtung zu Redundanz und Dimensionsstruktur

- Benachbarte verfügbare Bänder sind meist nahezu perfekt korreliert (Median 0,999). Niedrigere Werte entstehen vor allem an größeren Lücken der Spektralachse.
- Die größten univariaten Crop-Unterschiede liegen im sichtbaren Bereich um 600 bis 700 nm. Für Stage liegen die größten F-Werte im Übergang zum NIR um 760 bis 875 nm sowie im SWIR.
- Nach Standardisierung erklären bereits zwei Hauptkomponenten mehr als 90 % der spektralen Varianz. In der 2D-Projektion überlappen die Crop- und Stage-Gruppen aber sichtbar.

Das spricht für stark redundante Spektralinformation und motiviert spätere, sauber validierte Vergleiche von PCA oder Bandauswahl. Ein hoher ANOVA-F-Wert und eine sichtbare PCA-Struktur sind keine Leistungswerte und können auch mit den ungleichen Gruppen, AEZ oder Monat zusammenhängen.

## 7. Ausreißer und auffällige Spektren (#19)

**Frage:** Gibt es Spektren mit ungewöhnlich großen Änderungen zwischen direkt benachbarten, verfügbaren Bändern?

Ein solcher Sprung kann auf Rauschen, einen Mess- oder Verarbeitungsfehler hinweisen, kann aber auch ein legitimes Spektralmuster sein. Wir vergleichen deshalb jeden Sprung mit der typischen Verteilung desselben Bandpaares und markieren nur Kandidaten zur Sichtprüfung. Das Screening ist keine Regel zum automatischen Entfernen von Zeilen.

In [ ]:
wavelength_steps = np.diff(available_wavelengths)
continuous_steps = wavelength_steps <= 15
step_values = np.abs(np.diff(complete_spectra.to_numpy(), axis=1))[:, continuous_steps]
step_median = np.median(step_values, axis=0)
step_mad = np.median(np.abs(step_values - step_median), axis=0)
robust_step_scores = (step_values - step_median) / (1.4826 * step_mad)
maximum_step_score = np.nanmax(robust_step_scores, axis=1)
maximum_step_index = np.nanargmax(robust_step_scores, axis=1)
continuous_band_pairs = [
    f"{left}–{right}"
    for left, right, is_continuous in zip(
        available_bands[:-1], available_bands[1:], continuous_steps, strict=True
    )
    if is_continuous
]

jump_candidates = train.loc[complete_train.index, [AEZ_COL, MONTH_COL, CROP_COL, STAGE_COL]].copy()
jump_candidates["maximaler robuster Sprungscore"] = maximum_step_score
jump_candidates["auffälliges Bandpaar"] = [
    continuous_band_pairs[position] for position in maximum_step_index
]
jump_candidates = jump_candidates.nlargest(10, "maximaler robuster Sprungscore")
display(jump_candidates)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(maximum_step_score, bins=40, color="C3", alpha=0.8)
axes[0].set(
    xlabel="maximaler robuster Sprungscore",
    ylabel="Anzahl Spektren",
    title="Verteilung ungewöhnlicher lokaler Sprünge",
)
axes[0].grid(axis="y", alpha=0.3)

axes[1].plot(
    available_wavelengths,
    complete_spectra.mean().to_numpy(),
    color="black",
    linewidth=2,
    label="Mittelwert vollständiger Spektren",
)
for spectrum_id in jump_candidates.index[:5]:
    axes[1].plot(
        available_wavelengths,
        complete_train.loc[spectrum_id, available_bands],
        alpha=0.75,
        label=spectrum_id,
    )
axes[1].set(
    xlabel="Wellenlänge [nm]",
    ylabel="Reflektanz [%]",
    title="Fünf Screening-Kandidaten",
)
axes[1].legend(fontsize="x-small")
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

### Beobachtung zu ungewöhnlichen Spektralsprüngen

Das Screening markiert einzelne Spektren mit sehr großen lokalen Änderungen, insbesondere bei Bandpaaren um 1.000 bis 1.700 nm. Die Abbildung zeigt, dass sich die Kandidaten vom mittleren Verlauf unterscheiden können; sie zeigt aber keinen eindeutigen Messfehler. Die Kandidaten werden deshalb nur dokumentiert und nicht entfernt. Eine spätere Cleaning-Regel braucht eine fachliche Begründung und muss innerhalb der Trainingspipeline geprüft werden.

### Region um `X912`–`X925`

**Frage:** Was ist in der Übergangsregion um 900 nm tatsächlich gemessen und wo liegen Lücken?

Wir zeigen die Verfügbarkeit und die mittleren Crop-Spektren von 880 bis 1.000 nm. Damit lässt sich prüfen, ob der Verlauf an dieser Stelle aus Messwerten besteht oder durch vollständig fehlende Bänder unterbrochen wird.

In [ ]:
TRANSITION_START_NM = 880
TRANSITION_END_NM = 1000
transition_bands = [
    band
    for band in bands
    if TRANSITION_START_NM <= int(band.removeprefix("X")) <= TRANSITION_END_NM
]
transition_wavelengths = np.asarray(wavelengths(train[transition_bands]))
transition_summary = pd.DataFrame(
    {
        "Wellenlänge [nm]": transition_wavelengths,
        "Train: fehlend": train[transition_bands].isna().sum().to_numpy(),
        "Test: fehlend": test[transition_bands].isna().sum().to_numpy(),
    },
    index=transition_bands,
)
display(transition_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for crop, crop_data in train.groupby(CROP_COL, observed=True):
    axes[0].plot(
        transition_wavelengths,
        crop_data[transition_bands].mean().to_numpy(),
        label=f"{crop} (n={len(crop_data)})",
    )
axes[0].set(
    xlabel="Wellenlänge [nm]",
    ylabel="mittlere Reflektanz [%]",
    title="Crop-Spektren um 900 nm",
)
axes[0].legend(fontsize="small")
axes[0].grid(alpha=0.3)
axes[1].scatter(transition_wavelengths, transition_summary["Train: fehlend"], label="Train", s=28)
axes[1].scatter(
    transition_wavelengths,
    transition_summary["Test: fehlend"],
    label="Test",
    marker="x",
    s=40,
)
axes[1].set(
    xlabel="Wellenlänge [nm]",
    ylabel="fehlende Werte",
    title="Fehlende Werte um 900 nm",
)
axes[1].legend()
axes[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
save_doc_figure(fig, "spektren_900nm")

### Beobachtung zur Region um 900 nm

- Die Bänder `X912`, `X915` und `X923` sind in Train und Test vollständig belegt.
- Von `X925` bis `X973` fehlen sechs aufeinanderfolgende Bänder in beiden Dateien vollständig.
- Die mittleren Crop-Kurven verlaufen bis `X923` ohne sichtbaren Einzelbandsprung; erst die fehlende Bandgruppe unterbricht die Darstellung.

Damit gibt es keinen EDA-Hinweis, die verfügbaren Bänder bis `X923` zusätzlich zu entfernen. Die vollständig leere Gruppe `X925`–`X973` wird wie alle anderen leeren Bänder aus der Modellpipeline entfernt. Ihre technische Ursache bleibt ohne Sensormetadaten offen.

### Ähnlichste, nicht identische Spektren

**Frage:** Gibt es Paare verschiedener Trainingszeilen, deren Spektren ungewöhnlich ähnlich sind?

Wir standardisieren die verfügbaren, vollständigen Bandwerte nur für diese Distanzanalyse und suchen je Spektrum den nächsten Nachbarn. Exakte Duplikate werden ausgeschlossen. Kleine Distanzen liefern lediglich Kandidaten für eine Sichtprüfung; ohne räumliche Metadaten zeigen sie nicht, dass die Zeilen vom selben Feld stammen.

In [ ]:
scaled_complete_spectra = StandardScaler().fit_transform(complete_spectra)
nearest_neighbors = NearestNeighbors(n_neighbors=1, metric="euclidean")
neighbor_distances, neighbor_indices = nearest_neighbors.fit(scaled_complete_spectra).kneighbors()
spectrum_ids = complete_train.index.to_numpy()
near_duplicate_candidates = pd.DataFrame(
    {
        "Spektrum": spectrum_ids,
        "nächster Nachbar": spectrum_ids[neighbor_indices[:, 0]],
        "standardisierte Distanz": neighbor_distances[:, 0],
    }
)
near_duplicate_candidates = near_duplicate_candidates.query("`standardisierte Distanz` > 0")
near_duplicate_candidates["Paar"] = near_duplicate_candidates.apply(
    lambda row: tuple(sorted((row["Spektrum"], row["nächster Nachbar"]))),
    axis=1,
)
near_duplicate_candidates = near_duplicate_candidates.drop_duplicates("Paar").sort_values(
    "standardisierte Distanz"
)
for column in [AEZ_COL, MONTH_COL, CROP_COL, STAGE_COL]:
    values = complete_train[column]
    near_duplicate_candidates[f"{column} 1"] = near_duplicate_candidates["Spektrum"].map(values)
    near_duplicate_candidates[f"{column} 2"] = near_duplicate_candidates["nächster Nachbar"].map(
        values
    )
display(near_duplicate_candidates.head(15).drop(columns="Paar"))

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(near_duplicate_candidates["standardisierte Distanz"], bins=40, color="C4", alpha=0.8)
ax.set(
    xlabel="Distanz zum nächsten nicht identischen Spektrum",
    ylabel="Anzahl Spektren",
    title="Verteilung der nächsten spektralen Nachbarn",
)
ax.grid(axis="y", alpha=0.3)
plt.show()

### Beobachtung zu ähnlichsten Spektren

- Nach Ausschluss exakter Duplikate liegen die ähnlichsten Paare bei standardisierten Distanzen von etwa 0,54 bis 0,64.
- Die gezeigten Kandidaten teilen überwiegend AEZ, Monat, Crop und Stage. Einzelne sehr ähnliche Paare haben jedoch unterschiedliche Stage-Labels.
- Die Distanzverteilung ist kontinuierlich; aus ihr folgt kein natürlicher Schwellenwert, ab dem zwei Spektren sicher demselben Feld zugeordnet werden können.

Die Near-Duplicate-Prüfung liefert damit keine Grundlage, zusätzliche Zeilen zu entfernen oder eine Gruppenvalidierung festzulegen. Sie dokumentiert aber ein mögliches Optimismusrisiko eines zufälligen Splits. Koordinaten, Feld-IDs oder Bild-IDs wären nötig, um dieses Risiko direkt zu prüfen.

## 8. Synthese und Cleaning-Empfehlung (#20)

### Data Understanding: gesicherte Befunde und nächste Prüfungen

| Datenfrage | Mit den Daten beantwortet | Konsequenz für die weitere Arbeit |
| --- | --- | --- |
| Welche Eingaben sind nutzbar? | 67 von 198 Bändern sind in Train und Test vollständig leer; 131 Bänder enthalten mindestens eine Messung. | Vollständig leere Bänder gehören nicht in ein Modell. Zusätzliche Lücken müssen innerhalb einer Pipeline behandelt werden. |
| Wie verlässlich ist ein zufälliger Split? | Zwei Paare von Trainingszeilen haben exakt gleiche Eingaben. Räumliche Gruppen sind nicht verfügbar. | Exakte Duplikate vor dem gemeinsamen Split entfernen; eine räumliche Validierung bleibt ohne Feld- oder Bild-ID offen. |
| Wie ausgewogen sind die Ziele? | `rice`, `Harvest` und mehrere Crop-Stage-Paare sind selten; einige Paare fehlen in der Stichprobe. | Nach `Crop|Stage` stratifizieren und Balanced Accuracy sowie Macro-F1 berichten. |
| Enthalten Metadaten starke Information? | `AEZ` und `Month` sind mit Crop und Stage assoziiert; einzelne Kultur-Monat-Gruppen sind nahezu einstufig. | Modelle mit und ohne Metadaten getrennt und transparent vergleichen. |
| Gibt es spektrale Struktur? | Mittelwerte, Indizes, ANOVA und PCA zeigen Unterschiede, aber auch Überlappungen und starke Bandkorrelationen. | Rohbänder, Indizes, PCA und mögliche Bandauswahl nur als getrennte Pipeline-Varianten vergleichen. |
| Gibt es klare Ausreißer? | Das Sprung-Screening liefert Kandidaten, aber keinen gesicherten Fehlernachweis. | Keine automatische Entfernung; erst fachlich begründete, innerhalb des Trainings geprüfte Regel entwickeln. |

### Noch nicht durch diese EDA beantwortet

- Ob die Reflexion TOA- oder Oberflächenreflektanz ist und welche Schritte vor dem Challenge-Export erfolgten.
- Was die Stage-Labels fachlich exakt bedeuten und ob `Month` den tatsächlichen Aufnahmemonat beschreibt.
- Ob Beobachtungen demselben Feld oder Bildausschnitt zugeordnet werden können.
- Ob eine der explorativ sichtbaren Strukturen auf unbekannten Daten tatsächlich die Balanced Accuracy für `Crop` oder `Stage` verbessert.

Diese offenen Punkte bleiben in [den Fragen an die Dozenten](../docs/projekt/offene-fragen.md) und werden nicht aus den EDA-Plots abgeleitet.

### Pre-Processing-Empfehlung aus der EDA

Dieser Abschnitt übersetzt die EDA-Befunde in konkrete Schritte für die spätere Modellierung. Er trennt **erforderliche Datenaufbereitung** von **Kandidaten, deren Nutzen erst im Training per Cross-Validation geprüft werden muss**. Kein lernbarer Schritt darf auf allen 5.591 Zeilen oder mit den unbeschrifteten Testdaten angepasst werden: Split zuerst, danach werden Transformer nur auf dem jeweiligen Trainingsanteil gefittet.

#### Feste Reihenfolge für jede Modell-Pipeline

| Schritt | Konkrete Umsetzung | EDA- und Domänenbezug |
| --- | --- | --- |
| Rohdaten prüfen | Daten ausschließlich mit `load_train()` und `load_test()` laden; das Pandera-Schema prüft Spalten, Typen, Labels und nichtnegative Bandwerte. `id` nie als Merkmal verwenden. | Verhindert Scheinsignale durch Formatfehler oder eine Identifikationsnummer. |
| Exakte Trainingsduplikate entfernen | Die zwei Gruppen identischer Trainingszeilen vor dem gemeinsamen Split entfernen. Danach `Crop|Stage` für Holdout und CV-Folds stratifizieren. | Sonst kann dieselbe Messung in Training und Validierung vorkommen; seltene Crop-Stage-Paare bleiben möglichst in beiden Teilen vertreten. |
| Vollständig leere Bänder entfernen | Im Trainingsanteil feststellen, welche Bandspalten nur `NaN` enthalten, und sie entfernen. Aktuell sind dies 67 Bänder; 131 Bänder bleiben. | Ein vollständig leeres Band enthält keine Messinformation. |
| Einzelne Lücken spektral interpolieren | Nach dem Entfernen leerer Bänder fehlende Werte je Spektrum entlang der Wellenlänge linear interpolieren; an Rändern den nächsten Messwert verwenden. Für ein vollständig leeres Spektrum nur den im Trainingsanteil gelernten Bandmedian verwenden. | Benachbarte Bänder sind stark korreliert; der Verlauf desselben Spektrums ist plausibler als ein globaler Spaltenmittelwert. |
| Metadaten gezielt kodieren | `AEZ` per One-Hot-Encoding kodieren; unbekannte AEZ im späteren Datensatz ignorieren. `Month` zunächst numerisch verwenden. | AEZ-Codes haben keine sinnvolle metrische Distanz; Monat ist stark mit Labels assoziiert. |
| Modellabhängig skalieren | Für SVM, logistische Regression, MLP, PCA und Distanzverfahren die spektralen Merkmale und `Month` mit einem nur auf dem Training gelernten `StandardScaler` standardisieren. Für Baumverfahren keine Skalierung erzwingen. | Die Reflektanzen und Metadaten liegen auf unterschiedlichen Skalen; Baum-Splits sind skalenunabhängig. |
| Klassenungleichgewicht im Training behandeln | Für geeignete Modelle `class_weight="balanced"`, sonst bereitgestellte balancierte Gewichte je `Crop|Stage` verwenden. Keine Gewichte aus Validierung oder Test ableiten. | Reis, Harvest und mehrere Crop-Stage-Paare sind selten; Balanced Accuracy bewertet Klassen gleichgewichtet. |

Die Schritte zum Entfernen leerer Bänder, zur Interpolation und zur Metadatenkodierung gehören als erster Schritt in jede sklearn-`Pipeline`, damit sie in jedem CV-Fold ausschließlich die Trainingsdaten sehen.

#### Domänenspezifische Varianten: vergleichen, nicht blind übernehmen

| Kandidat | Konkreter Vergleich | Fachliche Begründung und Vorsicht |
| --- | --- | --- |
| Vegetationsindizes | Verfügbare Rohbänder, Rohbänder plus NDVI/NDRE/PRI/NDWI und nur Indizes vergleichen. Die Indizes nach der Imputation und innerhalb der Pipeline berechnen. | In der EDA unterscheiden sich NDVI, NDRE und NDWI zwischen Gruppen, überlappen aber. CAI nicht verwenden: `X2002` ist leer. |
| Zyklische Monatskodierung | Roher `Month` gegen `sin(2π·Month/12)` und `cos(2π·Month/12)` vergleichen; beide Varianten mit einer vollständig spektralbasierten Variante vergleichen. | Monate sind zyklisch, aber nur von Mai bis Oktober belegt; der Kalendereffekt kann fragil sein. |
| Spektrale Glättung | Keine Standardglättung. Falls geprüft, Savitzky-Golay-Glättung mit wenigen vorab festgelegten Fenster- und Polynomgraden nur innerhalb der CV testen. | Sie könnte Sensorrauschen reduzieren, kann aber schmale klassenrelevante Strukturen verwischen. |
| PCA oder Bandauswahl | Rohbänder gegen PCA-Varianten mit vorab festgelegter Komponentenanzahl und gegen eine Auswahl aus wenigen, nur im Trainingsfold bestimmten Bändern vergleichen. | Redundante Bänder und wenige PCA-Komponenten motivieren den Vergleich, sind aber kein Leistungsnachweis. |
| `AEZ` und `Month` als Eingaben | Jeden Modellansatz mit und ohne Metadaten auswerten und Fehler nach Crop, Stage, AEZ und Month berichten. | Metadaten können echten Kontext liefern, aber auch Shortcut-Lernen fördern. |
| Ausreißerbehandlung | Keine Zeilen allein wegen des Sprung-Screenings entfernen. Eine spätere Regel nur im Trainingsanteil festlegen und gegen die Variante ohne Entfernung vergleichen. | Die EDA markiert Kandidaten, weist aber keinen Messfehler nach. |

#### Nicht als Pre-Processing verwechseln

- Ein räumlicher oder feldweiser Split ist eine **Validierungsstrategie**, keine Merkmalsaufbereitung. Ohne Koordinaten oder Feld-IDs ist er derzeit nicht möglich.
- Das Erzwingen gültiger Crop-Stage-Paare ist eine **Entscheidung der Modellierung oder Nachverarbeitung**, keine Transformation der Eingaben.
- Jede Wahl zwischen den Kandidaten erfolgt über Cross-Validation im Trainingsanteil. Der 30-%-Holdout wird danach genau einmal für die gewählte Variante verwendet.

#### Empfohlene Startvarianten

1. **Referenz:** leere Bänder entfernen, Lücken interpolieren, Rohbänder; je nach Modell skalieren; ohne `AEZ` und `Month`.
2. **Kontextvergleich:** gleiche Pipeline mit One-Hot-AEZ und `Month`.
3. **Domänenvergleich:** Rohbänder plus die vier vorab definierten Indizes, jeweils mit und ohne Metadaten.
4. **Reduktionsvergleich:** beste bisherige Variante gegen PCA oder eine trainingsfold-basierte Bandauswahl.

Alle Varianten berichten Balanced Accuracy und Macro-F1 getrennt für `Crop` und `Stage`, zusätzlich Samples-F1 und den Anteil ungültiger Crop-Stage-Paare. Erst dieser Vergleich beantwortet, welche Pre-Processing-Schritte die Vorhersage tatsächlich verbessern.